# Rebuild the dataset for Hisar–Fatehabad–Sirsa

The first dataset came from Sentinel-2 tile **T43REM**, which is the Haryana–Rajasthan border (Bhiwani / Jhunjhunu / Churu).
Hisar, Fatehabad and Sirsa are in the tile just north of it, **T43REN**. This notebook rebuilds the dataset there,
using **the same pipeline scripts from the GitHub repo** (01 tiling → 02b SEN2SR → 06 centre crop → 05 metadata clean),
so the method is unchanged. Only the location (and how dates are picked) changes.

What it does, start to finish (~45-75 min, no manual downloads):
1. Finds every T43REN scene in the rabi window (1 Jan – 28 Feb) for 2020–2024 and measures cloud **inside our area**.
2. Picks one clear scene per year, closest to 27 January (same timing as before).
3. Downloads only our 85 × 70 km area from each scene.
4. Finds agricultural tiles and **keeps only those inside Hisar, Fatehabad or Sirsa districts**.
5. Cuts the same locations from all 5 dates, removes cloudy tiles automatically, super-resolves (SEN2SR) and centre-crops.
6. Checks the location again on a map and makes the collages.

**Settings (right panel):** Internet → **On**. Accelerator → **None** is fine (super-resolution runs on CPU, ~20 min);
choose GPU only if no other GPU notebook is running.
**Run with:** Save Version → **Save & Run All (Commit)**. Then download from the **Output** tab:
`hisar_dataset_final.zip`, `collages_hisar.zip`, `location_check.png`, `scene_previews.png`, `contact_sheet.png`.

In [ ]:
# 1. Install + get the pipeline scripts from GitHub
!pip install -q pystac-client rasterio pyproj shapely mlstac sen2sr
!rm -rf /tmp/repo && git clone -q --depth 1 https://github.com/japmanyakaur/ephemeral-gully-domain-adaptation /tmp/repo && ls /tmp/repo

In [ ]:
# 2. Settings
import os, glob, json, shutil, subprocess, sys
import numpy as np, pandas as pd, requests, rasterio
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from pyproj import Transformer
from shapely.geometry import shape, Point
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt

os.environ.update(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif,.TIF",
                  GDAL_HTTP_MAX_RETRY="5", GDAL_HTTP_RETRY_DELAY="3")

REPO  = "/tmp/repo"
WORK  = "/tmp/build"                       # temporary files (not saved)
OUT   = "/kaggle/working"                  # saved files
RAW   = f"{WORK}/raw"
TILES = f"{WORK}/tiles_10m"
SR    = f"{WORK}/tiles_sr"
FINAL = f"{OUT}/hisar_dataset_final"
COLL  = f"{OUT}/collages_hisar"

MGRS = "43REN"                                                        # tile containing Hisar, Fatehabad, Sirsa
CROP = dict(xmin=500000, xmax=585000, ymin=3215000, ymax=3285000)     # UTM 43N, 85 x 70 km
YEARS = [2020, 2021, 2022, 2023, 2024]
SEASON = ("01-01", "02-28")             # rabi window
TARGET_DAY = "01-27"                    # same timing as the first dataset
MAX_AREA_CLOUD = 0.02                   # max cloud/shadow fraction inside our area for a scene to count as clear
NUM_TILES = 300
KEEP_DISTRICTS = {"Hisar", "Fatehabad", "Sirsa"}
TILE_CLOUD_MAX = 0.05                   # drop a tile-date if >5% of its 320 m footprint is cloud/shadow
CLOUD_CLASSES = [3, 8, 9, 10, 11]       # SCL: shadow, cloud medium, cloud high, cirrus, snow
DISTRICTS_URL = "https://raw.githubusercontent.com/geohacker/india/master/district/india_district.geojson"
CITIES = {"Hisar": (75.7217, 29.1492), "Fatehabad": (75.4548, 29.5153), "Sirsa": (75.0280, 29.5349)}

for d in [WORK, RAW]:
    os.makedirs(d, exist_ok=True)
for d in [TILES, SR, FINAL, COLL]:
    shutil.rmtree(d, ignore_errors=True)
to_ll = Transformer.from_crs("EPSG:32643", "EPSG:4326", always_xy=True)
CROP_ARGS = ["--xmin", str(CROP["xmin"]), "--xmax", str(CROP["xmax"]),
             "--ymin", str(CROP["ymin"]), "--ymax", str(CROP["ymax"])]

def run(cmd, cwd=None):
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=cwd)
    print(r.stdout[-2500:])
    if r.returncode != 0:
        print(r.stderr[-4000:])
        raise RuntimeError("step failed: " + cmd[1])

def read_area(href, bounds=CROP, out_shape=None):
    # Read only our area from a (cloud-hosted) GeoTIFF. Returns (array, transform, profile)
    with rasterio.open(href) as s:
        w = from_bounds(bounds["xmin"], bounds["ymin"], bounds["xmax"], bounds["ymax"], s.transform)
        w = w.round_offsets().round_lengths()
        kw = dict(window=w, boundless=True, fill_value=0)
        if out_shape:
            kw.update(out_shape=(s.count, *out_shape), resampling=Resampling.average)
        a = s.read(**kw)
        return a, s.window_transform(w), s.profile.copy()

# district boundaries (to keep only Hisar / Fatehabad / Sirsa)
dj = requests.get(DISTRICTS_URL, timeout=300).json()
DISTRICTS = [(f["properties"]["NAME_2"], shape(f["geometry"])) for f in dj["features"]
             if f["properties"]["NAME_1"] == "Haryana"]
def district_of(lon, lat):
    p = Point(lon, lat)
    return next((name for name, g in DISTRICTS if g.contains(p)), "outside Haryana")
print("Haryana districts loaded:", len(DISTRICTS))

In [ ]:
# 3. Find scenes and pick one clear date per year
from pystac_client import Client
cat = Client.open("https://earth-search.aws.element84.com/v1")

rows = []
for y in YEARS:
    items = cat.search(collections=["sentinel-2-l2a"], datetime=f"{y}-{SEASON[0]}/{y}-{SEASON[1]}",
                       query={"grid:code": {"eq": f"MGRS-{MGRS}"}}).item_collection()
    for it in items:
        rows.append(dict(year=y, date=it.datetime.date().isoformat(), id=it.id,
                         scene_cloud=it.properties.get("eo:cloud_cover", 100),
                         tci=it.assets["visual"].href, scl=it.assets["scl"].href))
cands = (pd.DataFrame(rows).sort_values(["date", "id"]).drop_duplicates("date", keep="last")
         .reset_index(drop=True))
print(f"{len(cands)} scenes found for {MGRS}")

# measure cloud + missing data INSIDE our area (scene-wide numbers can hide a cloudy corner)
cands["days_from_target"] = cands.apply(
    lambda r: abs((pd.Timestamp(r.date) - pd.Timestamp(f"{r.year}-{TARGET_DAY}")).days), axis=1)
cands = cands[cands["scene_cloud"] <= 30].sort_values(["year", "days_from_target"]).groupby("year").head(8)
area_cloud, area_nodata = [], []
for _, r in cands.iterrows():
    a, _, _ = read_area(r.scl)
    a = a[0]
    valid = a > 0
    area_nodata.append(round(1 - valid.mean(), 4))
    area_cloud.append(round(np.isin(a, CLOUD_CLASSES).sum() / max(valid.sum(), 1), 4))
    print(f"  {r.date}  scene cloud {r.scene_cloud:5.1f}%  area cloud {area_cloud[-1]:.2%}  area missing {area_nodata[-1]:.2%}")
cands["area_cloud"], cands["area_nodata"] = area_cloud, area_nodata

chosen = []
for y, g in cands.groupby("year"):
    ok = g[(g.area_nodata < 0.01) & (g.area_cloud <= MAX_AREA_CLOUD)]
    pick = ok.sort_values("days_from_target").iloc[0] if len(ok) else \
           g.assign(bad=g.area_cloud + g.area_nodata).sort_values(["bad", "days_from_target"]).iloc[0]
    chosen.append(pick)
chosen = pd.DataFrame(chosen).reset_index(drop=True)
print("\nCHOSEN SCENES:")
print(chosen[["date", "id", "scene_cloud", "area_cloud", "area_nodata"]].to_string(index=False))
chosen.drop(columns=["tci", "scl"]).to_csv(f"{OUT}/scenes_chosen.csv", index=False)

In [ ]:
# 4. Download only our area from each chosen scene (+ preview picture)
raw_files = []
for _, r in chosen.iterrows():
    out = f"{RAW}/T{MGRS}_{r.date.replace('-', '')}T000000_TCI_10m.tif"   # name format 01_tile_sentinel2.py expects
    if not os.path.exists(out):
        a, tr, prof = read_area(r.tci)
        prof.update(driver="GTiff", width=a.shape[2], height=a.shape[1], count=a.shape[0], transform=tr,
                    compress="deflate", tiled=True, blockxsize=512, blockysize=512, nodata=0)
        for k in ["interleave", "photometric"]:
            prof.pop(k, None)
        with rasterio.open(out, "w", **prof) as dst:
            dst.write(a)
    raw_files.append(out)
    print("saved", os.path.basename(out), f"{os.path.getsize(out)/1e6:.0f} MB")

fig, axes = plt.subplots(1, len(raw_files), figsize=(4 * len(raw_files), 4))
for ax, f in zip(np.atleast_1d(axes), raw_files):
    with rasterio.open(f) as s:
        a = s.read(out_shape=(3, s.height // 10, s.width // 10), resampling=Resampling.average)
    ax.imshow(np.transpose(a, (1, 2, 0))); ax.set_title(os.path.basename(f)[7:15]); ax.axis("off")
plt.suptitle("Our area on each chosen date (85 x 70 km) - look for haze or cloud")
plt.tight_layout(); plt.savefig(f"{OUT}/scene_previews.png", dpi=110); plt.show()

In [ ]:
# 5. Find agricultural tiles on the latest date, keep only Hisar / Fatehabad / Sirsa districts
ref = raw_files[-1]
run([sys.executable, f"{REPO}/01_tile_sentinel2.py", "--input", ref, "--output", TILES,
     "--num-tiles", str(int(NUM_TILES * 1.6))] + CROP_ARGS)

meta = pd.read_csv(f"{TILES}/metadata.csv")
ref_date = meta["date"].iloc[0]
lon, lat = to_ll.transform((meta.utm_x + 640).values, (meta.utm_y - 640).values)   # tile centre
meta["district"] = [district_of(a, b) for a, b in zip(lon, lat)]
print("Candidate tiles by district:\n", meta["district"].value_counts().to_string())

keep = meta[meta["district"].isin(KEEP_DISTRICTS)].sort_values("tile_id")
if len(keep) > NUM_TILES:      # spread the pick over the whole list, like the original stride selection
    keep = keep.iloc[np.linspace(0, len(keep) - 1, NUM_TILES).round().astype(int)]
print(f"Keeping {len(keep)} tiles inside {sorted(KEEP_DISTRICTS)}")

# renumber 0..N-1 and remove the rest (two-step rename avoids name clashes)
img_dir = f"{TILES}/images/{ref_date}"
tmp = f"{WORK}/renumber"; shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
keep = keep.reset_index(drop=True)
for new_id, r in keep.iterrows():
    shutil.move(f"{img_dir}/{r.filename}", f"{tmp}/rgb_0_tile_{new_id}.png")
shutil.rmtree(img_dir); shutil.move(tmp, img_dir)
keep["tile_id"] = keep.index
keep["filename"] = [f"rgb_0_tile_{i}.png" for i in keep.index]
keep.drop(columns=["district"]).to_csv(f"{TILES}/metadata.csv", index=False)   # same columns 01 writes
print(len(os.listdir(img_dir)), "tiles saved for", ref_date)

In [ ]:
# 6. Cut the SAME locations from the other dates
for f in raw_files[:-1]:
    run([sys.executable, f"{REPO}/01_tile_sentinel2.py", "--input", f, "--output", TILES,
         "--reuse-locations", f"{TILES}/metadata.csv"] + CROP_ARGS)
meta = pd.read_csv(f"{TILES}/metadata.csv")
print(meta.groupby("date").size().to_string())

In [ ]:
# 7. Automatic cloud check for every tile-date (uses the satellite's own cloud mask, SCL)
#    Only the central 320 m matters, because that is what is kept after the centre crop.
removed = []
for _, sc in chosen.iterrows():
    scl, tr, _ = read_area(sc.scl)
    scl = scl[0]
    inv = ~tr
    for _, r in meta[meta["date"] == sc.date].iterrows():
        cx, cy = r.utm_x + 640, r.utm_y - 640
        c0, r0 = inv * (cx - 160, cy + 160)
        c1, r1 = inv * (cx + 160, cy - 160)
        patch = scl[int(round(r0)):int(round(r1)), int(round(c0)):int(round(c1))]
        valid = patch > 0
        bad = valid.mean() < 0.9 or np.isin(patch, CLOUD_CLASSES).sum() / max(valid.sum(), 1) > TILE_CLOUD_MAX
        if bad:
            os.remove(f"{TILES}/images/{r.date}/{r.filename}")
            removed.append((r.date, r.tile_id))
print(f"Removed {len(removed)} cloudy / incomplete tile-dates")
run([sys.executable, f"{REPO}/05_clean_metadata.py", "--dir", TILES])

In [ ]:
# 8. Super-resolution 10 m -> 2.5 m with SEN2SR (same script as before)
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def super_resolve():
    run([sys.executable, f"{REPO}/02b_super_resolve_sen2sr.py", "--input", TILES, "--output", SR,
         "--device", device], cwd=WORK)

super_resolve()

In [ ]:
# 9. Centre-crop 512 -> 128 px (320 m), then drop dark / blank tiles
run([sys.executable, f"{REPO}/06_center_crop.py", "--input", SR, "--output", FINAL, "--crop-size", "128"])

meta = pd.read_csv(f"{FINAL}/metadata.csv")
dark = 0
for _, r in meta.iterrows():
    p = f"{FINAL}/images/{r.date}/{r.filename}"
    a = np.asarray(Image.open(p).convert("RGB"), dtype=np.float32)
    if a.mean() < 15 or a.mean() > 235 or a.std() < 3:
        os.remove(p); dark += 1
print("Removed dark/blank tiles:", dark)
run([sys.executable, f"{REPO}/05_clean_metadata.py", "--dir", FINAL])

# add the real-world position of every tile, so the location can always be checked
meta = pd.read_csv(f"{FINAL}/metadata.csv")
lon, lat = to_ll.transform((meta.utm_x + 160).values, (meta.utm_y - 160).values)   # utm_x/y = top-left of 320 m tile
meta["lat"], meta["lon"] = np.round(lat, 6), np.round(lon, 6)
meta["district"] = [district_of(a, b) for a, b in zip(lon, lat)]
meta.to_csv(f"{FINAL}/metadata.csv", index=False)
print(meta.groupby("date").size().to_string())
print("Locations:", meta.tile_id.nunique(), " images:", len(meta))

In [ ]:
# 10. LOCATION CHECK - where are the tiles really?
loc = meta.drop_duplicates("tile_id")
print("Locations per district:\n", loc["district"].value_counts().to_string())
share = loc["district"].isin(KEEP_DISTRICTS).mean()
print(f"\n{share:.0%} of locations are in Hisar / Fatehabad / Sirsa")
assert share == 1.0, "Some tiles are outside the three districts - stop and check!"

fig, ax = plt.subplots(figsize=(9, 8))
for name, g in DISTRICTS:
    if name in KEEP_DISTRICTS or g.bounds[0] < 76.0:
        xs, ys = (g.geoms[0] if g.geom_type == "MultiPolygon" else g).exterior.xy
        ax.plot(xs, ys, color="0.5", lw=0.8)
        if name in KEEP_DISTRICTS:
            c = g.representative_point(); ax.text(c.x, c.y, name + " district", color="0.35", fontsize=10, ha="center", clip_on=True)
colors = {"Hisar": "#2b8a3e", "Fatehabad": "#1971c2", "Sirsa": "#e8590c"}
for d, g in loc.groupby("district"):
    ax.scatter(g.lon, g.lat, s=14, color=colors.get(d, "red"), label=f"{d} ({len(g)})")
for n, (x, y) in CITIES.items():
    ax.plot(x, y, "k*", ms=12); ax.annotate(n + " city", (x, y), xytext=(5, 5), textcoords="offset points", fontweight="bold", annotation_clip=True)
xs_all = list(loc.lon) + [v[0] for v in CITIES.values()]; ys_all = list(loc.lat) + [v[1] for v in CITIES.values()]
ax.set_xlim(min(xs_all) - 0.1, max(xs_all) + 0.1); ax.set_ylim(min(ys_all) - 0.1, max(ys_all) + 0.1)
ax.set_aspect("equal"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title(f"Study area check: {loc.tile_id.nunique()} locations, Sentinel-2 tile T{MGRS}")
plt.savefig(f"{OUT}/location_check.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# 11. Contact sheet (40 random locations, latest date) for a quick visual check
latest = meta["date"].max()
sample = meta[meta.date == latest].sample(min(40, (meta.date == latest).sum()), random_state=0)
fig, axes = plt.subplots(5, 8, figsize=(16, 10.5))
for ax in axes.flat:
    ax.axis("off")
for ax, (_, r) in zip(axes.flat, sample.iterrows()):
    ax.imshow(Image.open(f"{FINAL}/images/{r.date}/{r.filename}")); ax.set_title(f"{r.tile_id} {r.district}", fontsize=8)
plt.suptitle(f"Random tiles, {latest} (128 px = 320 m, 2.5 m/px)"); plt.tight_layout()
plt.savefig(f"{OUT}/contact_sheet.png", dpi=110); plt.show()

In [ ]:
# 12. Collages (same 2x3 layout as before) + index file for the VLM notebooks
from PIL import ImageDraw, ImageFont
CELL, COLS, ROWS, PAD = 384, 3, 2, 8

def get_font(size):
    for name in ["DejaVuSans-Bold.ttf", "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", "arialbd.ttf"]:
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            continue
    return ImageFont.load_default()

def make_collage(frames, legend, font, small_font):
    canvas = Image.new("RGB", (COLS * CELL + (COLS + 1) * PAD, ROWS * CELL + (ROWS + 1) * PAD), (255, 255, 255))
    draw = ImageDraw.Draw(canvas)
    for i in range(COLS * ROWS):
        r, c = divmod(i, COLS)
        x, y = PAD + c * (CELL + PAD), PAD + r * (CELL + PAD)
        if i >= len(frames):
            draw.rectangle([x, y, x + CELL, y + CELL], fill=(235, 235, 235))
            draw.multiline_text((x + CELL // 2, y + CELL // 2), legend, fill=(40, 40, 40), font=small_font,
                                anchor="mm", align="center", spacing=8)
            continue
        label, path = frames[i]
        if path and os.path.exists(path):
            canvas.paste(Image.open(path).convert("RGB").resize((CELL, CELL), Image.LANCZOS), (x, y))
        else:
            draw.rectangle([x, y, x + CELL, y + CELL], fill=(60, 60, 60))
            draw.text((x + CELL // 2, y + CELL // 2), "no image", fill="white", font=small_font, anchor="mm")
        bbox = draw.textbbox((x + 6, y + 6), label, font=font)
        draw.rectangle([bbox[0] - 4, bbox[1] - 3, bbox[2] + 4, bbox[3] + 3], fill=(0, 0, 0))
        draw.text((x + 6, y + 6), label, fill="white", font=font)
    return canvas

dates = sorted(meta["date"].unique())
os.makedirs(COLL, exist_ok=True)
font, small = get_font(30), get_font(28)
legend = f"Same field,\n{len(dates)} winters\n({dates[0][:4]}-{dates[-1][:4]})\n320 m x 320 m"
index = []
for tid, g in meta.groupby("tile_id"):
    have = dict(zip(g.date, g.filename))
    frames = [(d[:7], f"{FINAL}/images/{d}/{have[d]}" if d in have else None) for d in dates]
    make_collage(frames, legend, font, small).save(f"{COLL}/collage_{tid:03d}.png")
    index.append([tid, f"collage_{tid:03d}.png", len(have), ""])
pd.DataFrame(index, columns=["tile_id", "collage", "n_images", "label"]).to_csv(f"{COLL}/collages.csv", index=False)
print("collages:", len(index))
plt.figure(figsize=(9, 6)); plt.imshow(Image.open(f"{COLL}/collage_000.png")); plt.axis("off"); plt.show()

In [ ]:
# 13. Zip for download + summary
for name in ["hisar_dataset_final", "collages_hisar"]:
    shutil.make_archive(f"{OUT}/{name}", "zip", OUT, name)
    print(f"{name}.zip  {os.path.getsize(f'{OUT}/{name}.zip')/1e6:.1f} MB")

print("\n=== SUMMARY ===")
print("Tile      :", "T" + MGRS)
print("Dates     :", ", ".join(dates))
print("Locations :", meta.tile_id.nunique())
print("Images    :", len(meta))
print("Complete  :", (meta.groupby('tile_id').size() == len(dates)).sum(), "locations have all", len(dates), "dates")
print("Districts :", meta.drop_duplicates('tile_id')['district'].value_counts().to_dict())